<a href="https://colab.research.google.com/github/virusds7778-coder/HSE_Denis_Ivanov_HOMEWORK_2026_2027/blob/main/lesson_1_%D0%94%D0%97_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Условия задания

1. Сгенерировать с использованием функции `range` (случайный шаг от 3 до 5)
   массив, содержащий отсортированные числа от 10 до 250 млн. Можно использовать
   `randint` из модуля `random` для ещё большей рандомизации значений, но для
   целей алгоритма бинарного поиска значения в массиве должны быть отсортированы.
2. Сгенерировать с помощью list comprehensions и функции `randint` (встроенный
   модуль `random`) 10 случайных чисел.
3. Написать функцию для алгоритма линейного поиска.
4. Написать функцию для алгоритма бинарного поиска.
5. Проверить наличие ранее сгенерированных случайных чисел в массиве с помощью
   алгоритмов линейного и бинарного поиска, замерить время.

Используются только модули из стандартной библиотеки, ничего устанавливать не
требуется.

In [11]:
import random
import sys
import time

try:  # Colab работает на Linux, там доступен учёт памяти через resource
    import resource
except ImportError:  # на Windows модуля нет
    resource = None

# --- Конфигурация -----------------------------------------------------------
MIN_VALUE = 10
MAX_VALUE = 250_000_000   # 25_000_000 - быстрый прогон на маленькой выборке
MIN_STEP = 3
MAX_STEP = 5
TARGETS_COUNT = 10
CHUNK_SIZE = 1_000_000
SEED = 42                 # None - каждый раз новые числа

random.seed(SEED)


def max_rss_mb():
    '''Пик потребления памяти процессом в МБ либо None, если недоступно.'''
    if resource is None:
        return None
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


print('Python', sys.version.split()[0])

Python 3.13.15


## Пункт 1. Отсортированный массив через `range` со случайным шагом от 3 до 5

Массив собирается «кусками» по `CHUNK_SIZE` элементов:

* внутри куска шаг фиксированный (`step = random.randint(3, 5)`), кусок строится
  одним вызовом `range(current, stop, step)`;
* следующий кусок начинается на `random.randint(3, 5)` больше конца предыдущего.

Счётчик `current` только растёт, поэтому весь массив **строго возрастает**,
дубликатов нет и к нему применим бинарный поиск — это требование задания.

Такая реализация почти целиком выполняется на уровне C (внутри `range` и
`list.extend`), поэтому генерация десятков миллионов элементов занимает секунды,
а не десятки минут, как цикл `for` по Python-итерациям.

In [12]:
def build_sorted_array(min_value=MIN_VALUE, max_value=MAX_VALUE,
                       min_step=MIN_STEP, max_step=MAX_STEP,
                       chunk_size=CHUNK_SIZE):
    '''Массив отсортированных чисел из [min_value, max_value] со случайным шагом.'''
    values = []
    current = min_value
    while current <= max_value:
        step = random.randint(min_step, max_step)
        stop = min(current + chunk_size * step, max_value + 1)
        values.extend(range(current, stop, step))
        # случайный зазор между кусками, чтобы не появилось дубликатов
        current = stop + random.randint(min_step, max_step)
    return values

## Пункт 2. Десять случайных чисел через list comprehension и `randint`

In [13]:
def generate_targets(min_value, max_value, count):
    '''count случайных чисел из диапазона через list comprehension.'''
    return [random.randint(min_value, max_value) for _ in range(count)]

## Пункт 3. Линейный поиск

Сложность **O(n)**. Работает с любыми данными, сортировка не нужна.

In [14]:
def linear_search(data, target):
    '''Линейный поиск: индекс первого вхождения target либо -1.'''
    for index, value in enumerate(data):
        if value == target:
            return index
    return -1

## Пункт 4. Бинарный поиск

Сложность **O(log n)**. Требует отсортированного массива с доступом по индексу
за O(1) — условие обеспечено пунктом 1.

In [5]:
def binary_search(data, target):
    '''Бинарный поиск (итеративный): индекс target либо -1.'''
    left, right = 0, len(data) - 1
    while left <= right:
        middle = (left + right) // 2
        value = data[middle]
        if value == target:
            return middle
        if value < target:
            left = middle + 1
        else:
            right = middle - 1
    return -1

## Самопроверка алгоритмов

Прежде чем гонять поиск по 60-миллионному массиву, убеждаемся, что обе функции
корректны и согласованы: массив отсортирован, дубликатов нет, для каждого
элемента массива оба поиска возвращают индекс самого значения, для отсутствующих
значений (в том числе на границах диапазона) возвращают `-1`.

In [6]:
def self_test():
    '''Проверка корректности обоих алгоритмов на маленьком массиве.'''
    small = build_sorted_array(10, 1000, chunk_size=50)
    assert small == sorted(small), 'массив не отсортирован'
    assert len(small) == len(set(small)), 'в массиве есть дубликаты'
    for value in small:
        assert small[linear_search(small, value)] == value
        assert small[binary_search(small, value)] == value
    for value in (-100, 0, 11, 999999):
        assert linear_search(small, value) == -1
        assert binary_search(small, value) == -1
    print('self-test OK: линейный и бинарный поиск согласованы')


self_test()

self-test OK: линейный и бинарный поиск согласованы


## Пункт 5. Замер времени

`time.perf_counter()` — самое точные часы для замеров коротких интервалов.
После каждого поиска проверяется `assert lin_index == bin_index`, то есть оба
алгоритма обязаны отвечать одинаково.

In [8]:
def measure(func, data, target):
    '''Выполняет поиск и возвращает (индекс, время в секундах).'''
    start = time.perf_counter()
    index = func(data, target)
    return index, time.perf_counter() - start


def format_time(seconds):
    '''Человекочитаемое представление интервала.'''
    if seconds < 1e-3:
        return '%.1f мкс' % (seconds * 1e6)
    if seconds < 1:
        return '%.2f мс' % (seconds * 1e3)
    return '%.3f с' % seconds


def run_experiment(min_value=MIN_VALUE, max_value=MAX_VALUE,
                   targets_count=TARGETS_COUNT):
    '''Генерирует массив и сравнивает линейный и бинарный поиск с замерами.'''
    print('Генерация массива [%d..%d] со случайным шагом %d..%d ...'
          % (min_value, max_value, MIN_STEP, MAX_STEP))
    start = time.perf_counter()
    data = build_sorted_array(min_value, max_value)
    build_time = time.perf_counter() - start
    print('Массив построен: %d элементов за %s'
          % (len(data), format_time(build_time)))
    rss = max_rss_mb()
    if rss:
        print('Пик памяти процесса: %.0f МБ' % rss)

    targets = generate_targets(min_value, max_value, targets_count)
    print('Случайные числа:', ', '.join(map(str, targets)), '')

    header = '%12s | %12s | %12s | %15s | ускорение' % (
        'число', 'линейный', 'бинарный', 'результат')
    print(header)
    print('-' * len(header))

    total_linear = 0.0
    total_binary = 0.0
    for target in targets:
        lin_index, lin_time = measure(linear_search, data, target)
        bin_index, bin_time = measure(binary_search, data, target)
        total_linear += lin_time
        total_binary += bin_time
        assert lin_index == bin_index, 'расхождение поиска для %d' % target
        result = 'индекс %d' % lin_index if lin_index != -1 else 'не найдено'
        speedup = lin_time / bin_time if bin_time > 0 else float('inf')
        print('%12d | %12s | %12s | %15s | %.0fx' % (
            target, format_time(lin_time), format_time(bin_time),
            result, speedup))

    print('-' * len(header))
    print('%12s | %12s | %12s | %15s | %.0fx' % (
        'ИТОГО', format_time(total_linear), format_time(total_binary), '',
        total_linear / total_binary))
    return data, targets

### Запуск эксперимента

**Про память.** Массив из ~60,7 млн элементов занимает примерно 2,2 ГБ
(8 байт на указатель в списке плюс 28 байт на объект `int`). Бесплатный рантайм
Colab имеет ~12 ГБ RAM, так что массив помещается, но другие тяжёлые ячейки
лучше выгрузить. Если рантайм сбросят или хочется быстрый прогон — поменяйте
`MAX_VALUE` на `25_000_000` в ячейке конфигурации либо вызовите
`run_experiment(10, 25_000_000)`.

In [9]:
data, targets = run_experiment()

Генерация массива [10..250000000] со случайным шагом 3..5 ...
Массив построен: 61599950 элементов за 2.053 с
Пик памяти процесса: 2459 МБ
Случайные числа: 241007505, 156640937, 107213267, 97172691, 58873477, 37133154, 136774932, 132477159, 24403319, 202878738 
       число |     линейный |     бинарный |       результат | ускорение
------------------------------------------------------------------------
   241007505 |      5.370 с |     19.4 мкс |      не найдено | 276703x
   156640937 |      3.856 с |     22.9 мкс |      не найдено | 168142x
   107213267 |      3.856 с |     21.0 мкс |      не найдено | 183960x
    97172691 |      5.393 с |     23.4 мкс |      не найдено | 230774x
    58873477 |      3.886 с |     21.9 мкс |      не найдено | 177492x
    37133154 |    554.07 мс |     16.5 мкс |  индекс 8626623 | 33678x
   136774932 |      3.953 с |     21.8 мкс |      не найдено | 181243x
   132477159 |      5.402 с |     23.9 мкс |      не найдено | 225951x
    24403319 |      4.109 

### Контрольный опыт: цели, которые точно есть в массиве

Случайное число попадает в массив примерно с вероятностью 1/4 (средний шаг 4),
поэтому большинство строк таблицы выше — «не найдено». Чтобы убедиться, что
поиск находит значения корректно, возьмём элементы самого массива.

In [10]:
checks = [data[0], data[len(data) // 3], data[len(data) // 2], data[-1]]
print('%12s | %12s | %12s | %15s' % ('число', 'линейный', 'бинарный', 'результат'))
print('-' * 58)
for value in checks:
    lin_index, lin_time = measure(linear_search, data, value)
    bin_index, bin_time = measure(binary_search, data, value)
    print('%12d | %12s | %12s | индекс %d, %s' % (
        value, format_time(lin_time), format_time(bin_time), lin_index,
        'совпало' if lin_index == bin_index else 'РАСХОЖДЕНИЕ'))

       число |     линейный |     бинарный |       результат
----------------------------------------------------------
          10 |      2.7 мкс |     19.9 мкс | индекс 0, совпало
    83666666 |      1.279 с |     18.0 мкс | индекс 20533316, совпало
   123400054 |      1.918 с |     18.5 мкс | индекс 30799975, совпало
   249999997 |      5.309 с |     16.4 мкс | индекс 61599949, совпало


## Ожидаемые результаты

Ниже — реальный прогон на локальной машине (Windows, Python 3.12.7 AMD64).
В Colab абсолютные времена будут другими (другой CPU и объём кэша), но порядок
величин и вывод о преимуществе бинарного поиска сохранятся.

```
Массив построен: 60749942 элементов за 1.858 с
Случайные числа: 168074234, 139172127, 42348485, 216863104, 29955042, 194735300,
                 90305927, 187513204, 93586729, 22108490

   168074234 |      1.679 с |     16.1 мкс |     не найдено | 104262x
   139172127 |      1.592 с |     15.2 мкс |     не найдено | 104770x
    42348485 |      1.482 с |     13.6 мкс |     не найдено | 109003x
   216863104 |      1.473 с |     12.7 мкс |     не найдено | 116004x
    29955042 |    175.81 мс |     10.9 мкс | индекс 7391002  |  16130x
   194735300 |      1.548 с |     13.1 мкс |     не найдено | 118173x
    90305927 |      1.599 с |     12.7 мкс |     не найдено | 125923x
   187513204 |      1.121 с |     10.2 мкс | индекс 44702606 | 109941x
    93586729 |      1.538 с |     12.7 мкс |     не найдено | 121066x
    22108490 |    125.10 мс |     10.7 мкс | индекс 5277116  |  11691x
       ИТОГО |     12.334 с |    127.9 мкс |                |  96434x
```

| Показатель | Значение |
| --- | --- |
| Диапазон значений | 10 … 250 000 000 |
| Средний шаг массива | ~4 (случайный 3…5) |
| Размер массива | 60 749 942 элемента |
| Время генерации массива | 1,858 с |
| Найдено из 10 чисел | 3 |
| Суммарно, линейный поиск | 12,334 с (~1,23 с на число) |
| Суммарно, бинарный поиск | 127,9 мкс (~12,8 мкс на число) |
| Среднее ускорение | ≈ 9,6 · 10⁴ раз |

## Выводы

1. Линейный поиск — **O(n)**: в худшем случае просматриваются все 60,7 млн
   элементов, на практике ~1,5–1,7 с на одно число. Если значение стоит в начале
   массива, время падает пропорционально позиции (29 955 042 → 175,8 мс;
   22 108 490 → 125,1 мс).
2. Бинарный поиск — **O(log₂ n)**: при n = 60 749 942 это ⌈log₂ n⌉ = 26 итераций
   независимо от результата. Время стабильное, 10…16 мкс, от позиции не зависит.
3. Выигрыш бинарного поиска — порядка **ста тысяч раз** (12,334 с против
   127,9 мкс на десять чисел). Плата — требование отсортированных данных, а
   поддержка сортировки при вставках дорога.
4. Найдено 3 числа из 10 — это ожидаемо: при среднем шаге 4 в массив входит
   каждая четвёртая величина, вероятность «попадания» случайного числа ~1/4.
   Контрольный опыт выше, где цели взяты из самого массива, находит все значения.
5. Линейный поиск применим к несортированным данным и структурам без
   произвольного доступа (например, связным спискам); бинарный требует
   отсортированную последовательность с индексированием за O(1).